In [1]:
from pyspark.sql import SparkSession, functions as F

spark = SparkSession.builder.appName("silver").master("local[*]").getOrCreate()

BRONZE = "/home/work/data/bronze/"
SILVER = "/home/work/data/silver/"
META = ["ingestion_ts", "source_file", "batch_id"]

TABLES = ["customers", "geolocation", "order_items", "order_payments",
          "order_reviews", "orders", "products", "sellers", "category_translation"]

# Types cibles. Les colonnes absentes restent en string (identifiants, codes postaux, textes).
TYPES = {
    "orders": {"order_purchase_timestamp": "timestamp", "order_approved_at": "timestamp",
               "order_delivered_carrier_date": "timestamp",
               "order_delivered_customer_date": "timestamp",
               "order_estimated_delivery_date": "timestamp"},
    "order_items": {"order_item_id": "int", "shipping_limit_date": "timestamp",
                    "price": "double", "freight_value": "double"},
    "order_payments": {"payment_sequential": "int", "payment_installments": "int",
                       "payment_value": "double"},
    "order_reviews": {"review_score": "int", "review_creation_date": "timestamp",
                      "review_answer_timestamp": "timestamp"},
    "products": {"product_name_lenght": "int", "product_description_lenght": "int",
                 "product_photos_qty": "int", "product_weight_g": "double",
                 "product_length_cm": "double", "product_height_cm": "double",
                 "product_width_cm": "double"},
    "geolocation": {"geolocation_lat": "double", "geolocation_lng": "double"},
}

# Journal des règles : chaque règle appliquée y est consignée avec ses chiffres
journal = []
def consigner(regle, table, avant, apres, note=""):
    journal.append({"regle": regle, "table": table, "avant": avant, "apres": apres, "note": note})

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/09 11:54:30 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [2]:
def appliquer_r1(df, nom):
    """R1 : chaînes vides -> NULL, puis typage sécurisé avec try_cast."""
    types = TYPES.get(nom, {})
    colonnes = []
    for c in df.columns:
        if c in META:
            colonnes.append(F.col(c))
        elif c in types:
            colonnes.append(F.expr(f"try_cast(nullif(trim({c}), '') AS {types[c]})").alias(c))
        else:
            colonnes.append(F.expr(f"nullif(trim({c}), '')").alias(c))
    return df.select(*colonnes)

In [3]:
print(f"{'table':22} {'lignes':>9}  statut")
for nom in TABLES:
    brz = spark.read.parquet(BRONZE + nom)
    slv = appliquer_r1(brz, nom)
    cols = [c for c in brz.columns if c not in META]

    avant = brz.agg(*[F.sum(F.when(F.col(c).isNull() | (F.trim(F.col(c)) == ""), 1).otherwise(0)).alias(c)
                      for c in cols]).first()
    apres = slv.agg(*[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
                      for c in cols]).first()

    ecarts = [c for c in cols if avant[c] != apres[c]]
    n = slv.count()
    statut = "OK" if (not ecarts and n == brz.count()) else f"ECART : {ecarts}"
    print(f"{nom:22} {n:>9}  {statut}")
    consigner("R1", nom, brz.count(), n, "typage et chaînes vides -> NULL")

# Contrôle visuel des types
appliquer_r1(spark.read.parquet(BRONZE + "orders"), "orders").printSchema()

table                     lignes  statut


customers                  99441  OK


geolocation              1000163  OK


order_items               112650  OK
order_payments            103886  OK
order_reviews              99224  OK
orders                     99441  OK
products                   32951  OK
sellers                     3095  OK
category_translation          71  OK
root
 |-- order_id: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- order_purchase_timestamp: timestamp (nullable = true)
 |-- order_approved_at: timestamp (nullable = true)
 |-- order_delivered_carrier_date: timestamp (nullable = true)
 |-- order_delivered_customer_date: timestamp (nullable = true)
 |-- order_estimated_delivery_date: timestamp (nullable = true)
 |-- ingestion_ts: timestamp (nullable = true)
 |-- source_file: string (nullable = true)
 |-- batch_id: string (nullable = true)



In [4]:
cust = appliquer_r1(spark.read.parquet(BRONZE + "customers"), "customers")
sel = appliquer_r1(spark.read.parquet(BRONZE + "sellers"), "sellers")

for nom, df, st, ville in [("customers", cust, "customer_state", "customer_city"),
                           ("sellers", sel, "seller_state", "seller_city")]:
    print(f"--- {nom} ---")
    print("États distincts           :", df.select(st).distinct().count())
    print("États mal formés          :", df.filter(~F.col(st).rlike("^[A-Z]{2}$")).count())
    brut = df.select(ville).distinct().count()
    norm = df.select(F.lower(F.trim(F.col(ville)))).distinct().count()
    print(f"Villes distinctes (brut)  : {brut}")
    print(f"Villes distinctes (minuscules, sans espaces) : {norm}")

--- customers ---
États distincts           : 27
États mal formés          : 0
Villes distinctes (brut)  : 4119
Villes distinctes (minuscules, sans espaces) : 4119
--- sellers ---
États distincts           : 23
États mal formés          : 0
Villes distinctes (brut)  : 611
Villes distinctes (minuscules, sans espaces) : 611


In [5]:
def silver_translation():
    """R8 (partie 1) : ajoute les 2 traductions manquantes, en les marquant is_manual."""
    base = (appliquer_r1(spark.read.parquet(BRONZE + "category_translation"), "category_translation")
            .withColumn("is_manual", F.lit(False)))
    manuelles = (spark.createDataFrame(
            [("pc_gamer", "pc_gamer"),
             ("portateis_cozinha_e_preparadores_de_alimentos", "portable_kitchen_and_food_preparers")],
            ["product_category_name", "product_category_name_english"])
        .join(base, "product_category_name", "left_anti")      # sécurité : pas de doublon
        .withColumn("ingestion_ts", F.current_timestamp())
        .withColumn("source_file", F.lit("manual"))
        .withColumn("batch_id", F.lit("manual"))
        .withColumn("is_manual", F.lit(True)))
    return base.unionByName(manuelles)


def silver_products(translation):
    """R8 (partie 2) et R9."""
    p = appliquer_r1(spark.read.parquet(BRONZE + "products"), "products")
    t = translation.select("product_category_name", F.col("product_category_name_english").alias("category_en"))
    return (p
        .withColumn("is_category_unknown", F.col("product_category_name").isNull())        # indicateur
        .withColumn("product_category_name", F.coalesce("product_category_name", F.lit("unknown")))  # R8
        .join(t, "product_category_name", "left")                                         # left : on garde tout
        .withColumn("category_en", F.coalesce("category_en", F.lit("unknown")))
        .withColumn("product_weight_g",                                                    # R9
                    F.when(F.col("product_weight_g") == 0, F.lit(None).cast("double"))
                     .otherwise(F.col("product_weight_g"))))
    

In [6]:
translation = silver_translation()
products = silver_products(translation)

brz_p = appliquer_r1(spark.read.parquet(BRONZE + "products"), "products")

print("Traductions : ", translation.count(), "(attendu 73, dont 2 manuelles :",
      translation.filter("is_manual").count(), ")")
print("Produits    :", products.count(), "(attendu 32951 : la jointure ne doit pas dupliquer)")
print("Catégorie inconnue          :", products.filter("is_category_unknown").count(), "(attendu 610)")
print("category_en = 'unknown'     :", products.filter("category_en = 'unknown'").count(),
      "(attendu 610 : plus aucun produit non traduit)")
print("Poids = 0                   :", products.filter("product_weight_g = 0").count(), "(attendu 0)")
print("Poids NULL avant / après    :", brz_p.filter("product_weight_g IS NULL").count(), "/",
      products.filter("product_weight_g IS NULL").count(), "(attendu 2 / 6)")

Traductions :  73 (attendu 73, dont 2 manuelles : 2 )
Produits    : 32951 (attendu 32951 : la jointure ne doit pas dupliquer)
Catégorie inconnue          : 610 (attendu 610)
category_en = 'unknown'     : 610 (attendu 610 : plus aucun produit non traduit)
Poids = 0                   : 0 (attendu 0)
Poids NULL avant / après    : 2 / 6 (attendu 2 / 6)


In [8]:
consigner("R8", "category_translation", 71, translation.count(), "+2 traductions manuelles")
consigner("R8", "products", brz_p.count(), products.count(),
          f"{products.filter('is_category_unknown').count()} catégories -> unknown")
consigner("R9", "products", 4, 0, "poids 0 -> NULL")

translation.write.mode("overwrite").parquet(SILVER + "category_translation")
products.write.mode("overwrite").parquet(SILVER + "products")
cust.write.mode("overwrite").parquet(SILVER + "customers")
sel.write.mode("overwrite").parquet(SILVER + "sellers")
print("Écrit :", [n for n in ["category_translation", "products", "customers", "sellers"]])

Écrit : ['category_translation', 'products', 'customers', 'sellers']


In [10]:
for nom in ["category_translation", "products", "customers", "sellers"]:
    d = spark.read.parquet(SILVER + nom)
    print(f"{nom:22} lignes={d.count():>7}  colonnes={len(d.columns)}")

category_translation   lignes=     73  colonnes=6
products               lignes=  32951  colonnes=14
customers              lignes=  99441  colonnes=8
sellers                lignes=   3095  colonnes=7


In [11]:
SEUIL_SHIPPED_JOURS = 60
FENETRE_DEBUT = "2017-01-01"
FENETRE_FIN = "2018-09-01"          # exclu : la fenêtre va jusqu'au 2018-08-31

flag = lambda cond: F.coalesce(cond, F.lit(False))   # NULL (date manquante) -> False

orders_r1 = appliquer_r1(spark.read.parquet(BRONZE + "orders"), "orders")
FIN_DATASET = orders_r1.agg(F.max("order_purchase_timestamp")).first()[0]
print("Fin du dataset :", FIN_DATASET)

# R5 : une ligne par order_id (distinct) pour ne pas dupliquer les commandes
avec_produits = (spark.read.parquet(BRONZE + "order_items").select("order_id").distinct()
                 .withColumn("has_items", F.lit(True)))

achat, appro = F.col("order_purchase_timestamp"), F.col("order_approved_at")
transp, livr = F.col("order_delivered_carrier_date"), F.col("order_delivered_customer_date")
estim, statut = F.col("order_estimated_delivery_date"), F.col("order_status")

o = (orders_r1
    .join(avec_produits, "order_id", "left")
    .withColumn("has_items", F.coalesce("has_items", F.lit(False)))
    # R5
    .withColumn("is_sale", F.col("has_items") & ~statut.isin("canceled", "unavailable"))
    # R7
    .withColumn("is_shipped_stale",
                (statut == "shipped") & livr.isNull()
                & (F.datediff(F.lit(FIN_DATASET), achat) > SEUIL_SHIPPED_JOURS))
    # R11 : indicateurs d'incohérence
    .withColumn("flag_carrier_before_approval", flag(transp < appro))
    .withColumn("flag_delivery_before_carrier", flag(livr < transp))
    .withColumn("is_canceled_but_delivered", (statut == "canceled") & livr.isNotNull())
    # R11 : délais (NULL si la commande n'est pas livrée)
    .withColumn("delivery_days", F.round((F.unix_timestamp(livr) - F.unix_timestamp(achat)) / 86400, 2))
    .withColumn("delay_days", F.datediff(F.to_date(livr), F.to_date(estim)))   # > 0 : en retard
    .withColumn("is_late", F.col("delay_days") > 0)
    # R14
    .withColumn("in_analysis_window",
                (achat >= F.lit(FENETRE_DEBUT).cast("timestamp"))
                & (achat < F.lit(FENETRE_FIN).cast("timestamp")))
    # R6 : motif de quarantaine
    .withColumn("quarantine_reason",
                F.when((statut == "delivered") & (appro.isNull() | transp.isNull() | livr.isNull()),
                       "delivered_missing_date")
                 .when(~F.col("has_items") & statut.isin("created", "invoiced", "shipped"),
                       "no_items_inconsistent_status")))

quarantaine_orders = o.filter(F.col("quarantine_reason").isNotNull())
orders_silver = o.filter(F.col("quarantine_reason").isNull()).drop("quarantine_reason")

Fin du dataset : 2018-10-17 17:30:18


In [12]:
print("Commandes avant quarantaine       :", o.count(), "(attendu 99441)")
print("has_items = false                 :", o.filter(~F.col("has_items")).count(), "(attendu 775)")
print("carrier avant approbation         :", o.filter("flag_carrier_before_approval").count(), "(attendu 1359)")
print("livraison avant transporteur      :", o.filter("flag_delivery_before_carrier").count(), "(attendu 23)")
print("canceled mais livrée              :", o.filter("is_canceled_but_delivered").count(), "(attendu 6)")

print("\nQuarantaine par motif (attendu 23 et 8) :")
quarantaine_orders.groupBy("quarantine_reason").count().show(truncate=False)

n_silver, n_quar = orders_silver.count(), quarantaine_orders.count()
print("Silver + quarantaine =", n_silver, "+", n_quar, "=", n_silver + n_quar, "(attendu 99441)")
print("Silver sans produit               :", orders_silver.filter(~F.col("has_items")).count(), "(attendu 767)")

print("\nIndicateurs à documenter (pas de valeur attendue) :")
print("is_shipped_stale                  :", orders_silver.filter("is_shipped_stale").count())
print("in_analysis_window = false        :", orders_silver.filter(~F.col("in_analysis_window")).count())
print("is_sale = true                    :", orders_silver.filter("is_sale").count())

print("\nRetard (commandes livrées uniquement) :")
orders_silver.filter(F.col("delay_days").isNotNull()).groupBy("is_late").count().show()

Commandes avant quarantaine       : 99441 (attendu 99441)
has_items = false                 : 775 (attendu 775)
carrier avant approbation         : 1359 (attendu 1359)
livraison avant transporteur      : 23 (attendu 23)
canceled mais livrée              : 6 (attendu 6)

Quarantaine par motif (attendu 23 et 8) :
+----------------------------+-----+
|quarantine_reason           |count|
+----------------------------+-----+
|delivered_missing_date      |23   |
|no_items_inconsistent_status|8    |
+----------------------------+-----+

Silver + quarantaine = 99410 + 31 = 99441 (attendu 99441)
Silver sans produit               : 767 (attendu 767)

Indicateurs à documenter (pas de valeur attendue) :
is_shipped_stale                  : 1098
in_analysis_window = false        : 346
is_sale = true                    : 98176

Retard (commandes livrées uniquement) :


[Stage 371:===========>                                             (1 + 4) / 5]

+-------+-----+
|is_late|count|
+-------+-----+
|   true| 6534|
|  false|89927|
+-------+-----+



In [13]:
consigner("R5", "orders", o.count(), o.filter("is_sale").count(), "is_sale = produits et statut hors canceled/unavailable")
consigner("R6", "orders", o.count(), n_silver, f"{n_quar} commandes en quarantaine")
consigner("R7", "orders", o.filter("status = 'shipped'" if False else "order_status = 'shipped'").count(),
          orders_silver.filter("is_shipped_stale").count(), f"shipped sans livraison de plus de {SEUIL_SHIPPED_JOURS} j")
consigner("R11", "orders", n_silver, n_silver, "indicateurs de dates et délais ajoutés")
consigner("R14", "orders", n_silver, orders_silver.filter("in_analysis_window").count(), "fenêtre 2017-01 à 2018-08")

orders_silver.write.mode("overwrite").parquet(SILVER + "orders")
quarantaine_orders.write.mode("overwrite").parquet(SILVER + "quarantine_orders")
print("Écrit : orders, quarantine_orders")

Écrit : orders, quarantine_orders


In [14]:
from pyspark.sql import Window

ids_silver = orders_silver.select("order_id")

def separer(df):
    """Sépare les lignes de détail : commandes présentes en Silver / commandes en quarantaine."""
    return (df.join(ids_silver, "order_id", "left_semi"),
            df.join(ids_silver, "order_id", "left_anti"))

# order_items : R1 puis séparation
items_r1 = appliquer_r1(spark.read.parquet(BRONZE + "order_items"), "order_items")
items_silver, items_quar = separer(items_r1)

# order_payments : R10
pay_r1 = appliquer_r1(spark.read.parquet(BRONZE + "order_payments"), "order_payments")
pay = (pay_r1
    .withColumn("is_zero_value", F.col("payment_value") <= 0)
    .withColumn("is_zero_installments", F.col("payment_installments") <= 0)
    .withColumn("is_undefined_type", F.col("payment_type") == "not_defined")
    .withColumn("payment_type",
                F.when(F.col("payment_type") == "not_defined", "unknown").otherwise(F.col("payment_type")))
    .withColumn("is_anomalous_payment",
                F.col("is_zero_value") | F.col("is_zero_installments") | F.col("is_undefined_type")))
pay_silver, pay_quar = separer(pay)

# order_reviews : R2 puis R3
rev_r1 = appliquer_r1(spark.read.parquet(BRONZE + "order_reviews"), "order_reviews")
w = Window.partitionBy("order_id").orderBy(F.col("review_answer_timestamp").desc(),
                                           F.col("review_id").desc())
rev = (rev_r1
    .withColumn("rn", F.row_number().over(w)).filter("rn = 1").drop("rn")          # R2
    .withColumn("has_comment",                                                      # R3
                F.col("review_comment_title").isNotNull() | F.col("review_comment_message").isNotNull()))
reviews_silver, reviews_quar = separer(rev)

In [15]:
# order_items
print("order_items : Silver + quarantaine =", items_silver.count(), "+", items_quar.count(),
      "=", items_silver.count() + items_quar.count(), "(attendu 112650)")
ca_total = items_r1.agg(F.sum("price")).first()[0]
ca_silver = items_silver.agg(F.sum("price")).first()[0]
print(f"CA avant séparation : {ca_total:,.2f} (attendu 13 591 643,70)")
print(f"CA Silver           : {ca_silver:,.2f}")
print(f"CA en quarantaine   : {ca_total - ca_silver:,.2f}")

# order_payments (comptages avant séparation, comparables à l'audit)
print("\norder_payments : lignes", pay.count(), "(attendu 103886)")
print("  paiements <= 0         :", pay.filter("is_zero_value").count(), "(attendu 9)")
print("  0 échéance             :", pay.filter("is_zero_installments").count(), "(attendu 2)")
print("  not_defined -> unknown :", pay.filter("is_undefined_type").count(), "(attendu 3)")
print("  lignes anormales       :", pay.filter("is_anomalous_payment").count(), "(attendu 11)")
print("  Silver + quarantaine   :", pay_silver.count(), "+", pay_quar.count())

# order_reviews
print("\norder_reviews : après R2 =", rev.count(), "(attendu 98673)")
print("  order_id distincts     :", rev.select("order_id").distinct().count(), "(doit égaler la ligne précédente)")
print("  has_comment = false    :", rev.filter(~F.col("has_comment")).count())
print("  Silver + quarantaine   :", reviews_silver.count(), "+", reviews_quar.count())

order_items : Silver + quarantaine = 112626 + 24 = 112650 (attendu 112650)
CA avant séparation : 13,591,643.70 (attendu 13 591 643,70)
CA Silver           : 13,588,545.09
CA en quarantaine   : 3,098.61

order_payments : lignes 103886 (attendu 103886)
  paiements <= 0         : 9 (attendu 9)
  0 échéance             : 2 (attendu 2)
  not_defined -> unknown : 3 (attendu 3)
  lignes anormales       : 11 (attendu 11)
  Silver + quarantaine   : 103855 + 31



order_reviews : après R2 = 98673 (attendu 98673)


  order_id distincts     : 98673 (doit égaler la ligne précédente)


  has_comment = false    : 56184


  Silver + quarantaine   : 98644 + 29


In [16]:
consigner("R2", "order_reviews", rev_r1.count(), rev.count(), "un avis par commande (le plus récent)")
consigner("R3", "order_reviews", rev.count(), rev.filter("has_comment").count(), "has_comment = true")
consigner("R10", "order_payments", pay.count(), pay.filter("is_anomalous_payment").count(),
          "lignes signalées, aucune supprimée")

for nom, silv, quar in [("order_items", items_silver, items_quar),
                        ("order_payments", pay_silver, pay_quar),
                        ("order_reviews", reviews_silver, reviews_quar)]:
    silv.write.mode("overwrite").parquet(SILVER + nom)
    quar.write.mode("overwrite").parquet(SILVER + "quarantine_" + nom)
    consigner("QUARANTAINE", nom, silv.count() + quar.count(), silv.count(),
              f"{quar.count()} lignes de commandes en quarantaine")

print("Écrit : order_items, order_payments, order_reviews et leurs tables quarantine_*")

26/11/09 12:58:45 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
26/11/09 12:58:51 WARN MemoryManager: Total allocation exceeds 95.00% (1,020,054,720 bytes) of heap memory
Scaling row group sizes to 95.00% for 8 writers
                                                                                

Écrit : order_items, order_payments, order_reviews et leurs tables quarantine_*


In [17]:
LAT_MIN, LAT_MAX, LNG_MIN, LNG_MAX = -34, 6, -74, -34

geo_r1 = appliquer_r1(spark.read.parquet(BRONZE + "geolocation"), "geolocation")

geo_dedup = geo_r1.drop(*META).dropDuplicates()
geo_ok = geo_dedup.filter(F.col("geolocation_lat").between(LAT_MIN, LAT_MAX)
                          & F.col("geolocation_lng").between(LNG_MIN, LNG_MAX))

geo_silver = (geo_ok.groupBy("geolocation_zip_code_prefix")
    .agg(F.avg("geolocation_lat").alias("lat"),
         F.avg("geolocation_lng").alias("lng"),
         F.count("*").alias("n_points"))
    .withColumnRenamed("geolocation_zip_code_prefix", "zip_code_prefix"))

In [18]:
print("Lignes Bronze              :", geo_r1.count(), "(attendu 1000163)")
print("Après dédoublonnage        :", geo_dedup.count(), "(attendu 738332 = 1000163 - 261831)")
print("Points hors Brésil restants:", geo_dedup.filter(~(F.col("geolocation_lat").between(LAT_MIN, LAT_MAX)
      & F.col("geolocation_lng").between(LNG_MIN, LNG_MAX))).count(), "(à comparer au 42 de l'audit)")
print("Codes postaux avant filtre :", geo_dedup.select("geolocation_zip_code_prefix").distinct().count(), "(attendu 19015)")
print("Codes postaux Silver       :", geo_silver.count())
print("Doublons de code postal    :", geo_silver.count() - geo_silver.select("zip_code_prefix").distinct().count(), "(attendu 0)")
print("Latitude/longitude NULL    :", geo_silver.filter("lat IS NULL OR lng IS NULL").count(), "(attendu 0)")

Lignes Bronze              : 1000163 (attendu 1000163)


Après dédoublonnage        : 738332 (attendu 738332 = 1000163 - 261831)


Points hors Brésil restants: 33 (à comparer au 42 de l'audit)


Codes postaux avant filtre : 19015 (attendu 19015)


Codes postaux Silver       : 19010


Doublons de code postal    : 0 (attendu 0)


[Stage 717:==========================================>              (6 + 2) / 8]

Latitude/longitude NULL    : 0 (attendu 0)


In [19]:
g = geo_silver.select("zip_code_prefix", "lat", "lng")

# On repart du Bronze pour éviter de lire et réécrire le même dossier Silver
cust_base = appliquer_r1(spark.read.parquet(BRONZE + "customers"), "customers")
sel_base = appliquer_r1(spark.read.parquet(BRONZE + "sellers"), "sellers")

cust_geo = (cust_base.join(g, cust_base.customer_zip_code_prefix == g.zip_code_prefix, "left")
            .withColumn("has_geo", F.col("lat").isNotNull())
            .withColumnRenamed("lat", "customer_lat").withColumnRenamed("lng", "customer_lng")
            .drop("zip_code_prefix"))
sel_geo = (sel_base.join(g, sel_base.seller_zip_code_prefix == g.zip_code_prefix, "left")
           .withColumn("has_geo", F.col("lat").isNotNull())
           .withColumnRenamed("lat", "seller_lat").withColumnRenamed("lng", "seller_lng")
           .drop("zip_code_prefix"))

print("customers :", cust_geo.count(), "(attendu 99441, aucun fan-out)")
print("  sans géolocalisation :", cust_geo.filter(~F.col("has_geo")).count(), "(278 d'après l'audit)")
print("sellers   :", sel_geo.count(), "(attendu 3095)")
print("  sans géolocalisation :", sel_geo.filter(~F.col("has_geo")).count(), "(7 d'après l'audit)")

customers : 99441 (attendu 99441, aucun fan-out)


  sans géolocalisation : 279 (278 d'après l'audit)
sellers   : 3095 (attendu 3095)


  sans géolocalisation : 7 (7 d'après l'audit)


In [20]:
consigner("R4", "geolocation", geo_r1.count(), geo_silver.count(), "dédoublonnage, filtre Brésil, 1 ligne par code postal")
consigner("R12", "customers", cust_geo.count(), cust_geo.filter("has_geo").count(), "left join géolocalisation")
consigner("R12", "sellers", sel_geo.count(), sel_geo.filter("has_geo").count(), "left join géolocalisation")

geo_silver.write.mode("overwrite").parquet(SILVER + "geolocation")
cust_geo.write.mode("overwrite").parquet(SILVER + "customers")
sel_geo.write.mode("overwrite").parquet(SILVER + "sellers")
print("Écrit : geolocation, customers, sellers (mis à jour)")

Écrit : geolocation, customers, sellers (mis à jour)


In [21]:
perdus = (geo_dedup.select("geolocation_zip_code_prefix").distinct()
          .join(geo_silver.select(F.col("zip_code_prefix").alias("geolocation_zip_code_prefix")),
                "geolocation_zip_code_prefix", "left_anti"))
perdus.show()
print(cust_geo.join(perdus, cust_geo.customer_zip_code_prefix == perdus.geolocation_zip_code_prefix,
                    "left_semi").count(), "client(s) dans ces codes")

+---------------------------+
|geolocation_zip_code_prefix|
+---------------------------+
|                      78131|
|                      83252|
|                      18243|
|                      53990|
|                      95130|
+---------------------------+



1 client(s) dans ces codes


In [22]:
def lire_silver(nom): return spark.read.parquet(SILVER + nom)

o = lire_silver("orders")
c = (lire_silver("customers")
     .select("customer_id", "customer_unique_id", "customer_state", "customer_city",
             F.col("has_geo").alias("customer_has_geo")))

it = (lire_silver("order_items").groupBy("order_id").agg(
    F.count("*").alias("n_items"),
    F.countDistinct("product_id").alias("n_products"),
    F.countDistinct("seller_id").alias("n_sellers"),
    F.round(F.sum("price"), 2).alias("items_total"),
    F.round(F.sum("freight_value"), 2).alias("freight_total")))

pa = (lire_silver("order_payments").groupBy("order_id").agg(
    F.count("*").alias("n_payments"),
    F.round(F.sum("payment_value"), 2).alias("payment_total"),
    F.max("payment_installments").alias("max_installments"),
    F.array_join(F.array_sort(F.collect_set("payment_type")), ",").alias("payment_types"),
    F.max("is_anomalous_payment").alias("has_anomalous_payment")))

rv = lire_silver("order_reviews").select("order_id", "review_score", "has_comment",
                                         "review_creation_date", "review_answer_timestamp")

enriched = (o.join(c, "customer_id", "left")
             .join(it, "order_id", "left")
             .join(pa, "order_id", "left")
             .join(rv, "order_id", "left")
             .withColumn("has_payment", F.col("n_payments").isNotNull())
             .withColumn("has_review", F.col("review_score").isNotNull())
             .fillna({"n_items": 0, "n_products": 0, "n_sellers": 0, "items_total": 0.0,
                      "freight_total": 0.0, "n_payments": 0, "payment_total": 0.0}))

enriched.write.mode("overwrite").parquet(SILVER + "orders_enriched")
print("Écrit : orders_enriched")

26/11/09 13:05:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

Écrit : orders_enriched


In [23]:
e = lire_silver("orders_enriched")
n = e.count()
print("Lignes                 :", n, "(attendu 99410)")
print("order_id distincts     :", e.select("order_id").distinct().count(), "(doit égaler la ligne précédente)")
print("Somme de n_items       :", e.agg(F.sum("n_items")).first()[0], "(attendu 112626 : aucun fan-out)")
print("Sans client            :", e.filter("customer_unique_id IS NULL").count(), "(attendu 0)")
print("Sans produit           :", e.filter("n_items = 0").count(), "(attendu 767)")
print("Sans paiement          :", e.filter("NOT has_payment").count())
print("Sans avis              :", e.filter("NOT has_review").count())

print("\nIntégrité référentielle (attendu 0 partout) :")
ids = lire_silver("orders").select("order_id")
for nom in ["order_items", "order_payments", "order_reviews"]:
    print(f"  {nom:16} -> orders :", lire_silver(nom).join(ids, "order_id", "left_anti").count())
print("  orders           -> customers :",
      lire_silver("orders").join(lire_silver("customers"), "customer_id", "left_anti").count())

# Mesure, sans valeur attendue : paiement vs produits + frais
ecart = (e.filter("n_items > 0 AND has_payment")
          .withColumn("ecart", F.round(F.col("payment_total") - F.col("items_total") - F.col("freight_total"), 2)))
print("\nCommandes dont le paiement diffère de produits + frais (> 1 centime) :",
      ecart.filter("abs(ecart) > 0.01").count(), "sur", ecart.count())

Lignes                 : 99410 (attendu 99410)
order_id distincts     : 99410 (doit égaler la ligne précédente)
Somme de n_items       : 112626 (attendu 112626 : aucun fan-out)
Sans client            : 0 (attendu 0)
Sans produit           : 767 (attendu 767)
Sans paiement          : 1
Sans avis              : 766

Intégrité référentielle (attendu 0 partout) :
  order_items      -> orders : 0
  order_payments   -> orders : 0
  order_reviews    -> orders : 0
  orders           -> customers : 0

Commandes dont le paiement diffère de produits + frais (> 1 centime) : 303 sur 98642


In [24]:
ca_source = (appliquer_r1(spark.read.parquet(BRONZE + "order_items"), "order_items")
             .agg(F.sum("price")).first()[0])
ca_silver = e.agg(F.sum("items_total")).first()[0]
ca_ventes = e.filter("is_sale").agg(F.sum("items_total")).first()[0]

print(f"CA source (Bronze, toutes lignes)          : {ca_source:>15,.2f}")
print(f"  - lignes de commandes en quarantaine      : {ca_source - ca_silver:>15,.2f}")
print(f"= CA Silver                                 : {ca_silver:>15,.2f}")
print(f"  - commandes canceled / unavailable        : {ca_silver - ca_ventes:>15,.2f}")
print(f"= CA des ventes (is_sale = true)            : {ca_ventes:>15,.2f}")

CA source (Bronze, toutes lignes)          :   13,591,643.70
  - lignes de commandes en quarantaine      :        3,098.61
= CA Silver                                 :   13,588,545.09
  - commandes canceled / unavailable        :       97,242.96
= CA des ventes (is_sale = true)            :   13,491,302.13


In [25]:
lignes = [(j["regle"], j["table"], j["avant"], j["apres"], j["note"]) for j in journal]
jr = spark.createDataFrame(lignes, ["regle", "table", "avant", "apres", "note"])
jr.coalesce(1).write.mode("overwrite").json(SILVER + "_journal")
jr.show(50, truncate=False)

print("\nInventaire de la couche Silver :")
for nom in ["customers", "sellers", "products", "category_translation", "geolocation",
            "orders", "order_items", "order_payments", "order_reviews", "orders_enriched",
            "quarantine_orders", "quarantine_order_items", "quarantine_order_payments",
            "quarantine_order_reviews"]:
    d = lire_silver(nom)
    print(f"  {nom:28} lignes={d.count():>8}  colonnes={len(d.columns)}")

+-----------+--------------------+-------+-------+------------------------------------------------------+
|regle      |table               |avant  |apres  |note                                                  |
+-----------+--------------------+-------+-------+------------------------------------------------------+
|R1         |customers           |99441  |99441  |typage et chaînes vides -> NULL                       |
|R1         |geolocation         |1000163|1000163|typage et chaînes vides -> NULL                       |
|R1         |order_items         |112650 |112650 |typage et chaînes vides -> NULL                       |
|R1         |order_payments      |103886 |103886 |typage et chaînes vides -> NULL                       |
|R1         |order_reviews       |99224  |99224  |typage et chaînes vides -> NULL                       |
|R1         |orders              |99441  |99441  |typage et chaînes vides -> NULL                       |
|R1         |products            |32951  |3295